# Performance and Valuation Insights: Bayer 04 Leverkusen's 2023/24 Bundesliga Season

This notebook presents a portfolio-style football analytics case study centered on **Bayer 04 Leverkusen's historic 2023/24 Bundesliga campaign**.

It focuses on three questions:

- *How dominant was Bayer 04 Leverkusen in Bundesliga 2023/24?*
- *Which players stood out in goals, assists, minutes, and market value?*
- *How much of player market value can be explained by simple season-level performance signals?*

The workflow combines **DuckDB**, **pandas**, **Matplotlib/Seaborn**, and **scikit-learn**.  
To improve reproducibility, the notebook reads the Transfermarkt dataset **remotely via DuckDB** instead of relying on a local `data/raw/` folder.

### Author: Moritz Philipp Haaf, BSc MA
**Email:** itzmore.dev@gmail.com

**GitHub Repo:** [itzmore-mph/bundesliga-performance-analysis](https://github.com/itzmore-mph/bundesliga-performance-analysis)

## Executive Summary

This notebook analyzes Bayer 04 Leverkusen's 2023/24 Bundesliga season through a reproducible football analytics workflow.

It is organized around one clear analytical scope:
- **competition:** Bundesliga (`L1`)
- **season window:** 2023/08/01 to 2024/05/31
- **club:** Bayer 04 Leverkusen Fußball

The notebook is structured for GitHub readability and reproducibility:
- one central configuration block,
- remote data access through DuckDB,
- season-scoped filtering,
- consistent variable naming,
- and modeling steps that can be rerun top to bottom.

## 1. Initial Setup

Import libraries, define notebook configuration, and prepare a project root for optional figure exports.

In [ ]:
# 1. Initial Setup

from pathlib import Path
import warnings
import re

import duckdb
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.dates as mdates

from adjustText import adjust_text
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="talk")

def find_project_root(start_path: Path) -> Path:
    current = start_path.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "README.md").exists() or (candidate / ".git").exists():
            return candidate
    return current

PROJECT_ROOT = find_project_root(Path.cwd())
FIGURES_DIR = PROJECT_ROOT / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

SEASON_LABEL = "2023/24"
SEASON_START = pd.Timestamp("2023-08-01")
SEASON_END = pd.Timestamp("2024-05-31")

TARGET_COMPETITION_ID = "L1"
TARGET_CLUB_NAME = "Bayer 04 Leverkusen Fußball"
DISPLAY_CLUB_NAME = "Bayer 04 Leverkusen"

BASE_URL = "https://pub-e682421888d945d684bcae8890b0ec20.r2.dev/data"

print(f"Project root: {PROJECT_ROOT}")
print(f"Figures directory: {FIGURES_DIR}")
print(f"Remote data source: {BASE_URL}")

: 

## 2. Load and Preprocess Data with pandas

Load the core Transfermarkt tables remotely via DuckDB, standardize column names, parse dates, validate key relationships, and build season-specific analysis subsets.

In [ ]:
# 2.1 Load remote tables via DuckDB
TABLE_NAMES = [
    "appearances",
    "club_games",
    "clubs",
    "competitions",
    "game_events",
    "games",
    "player_valuations",
    "players",
]

remote_con = duckdb.connect(database=":memory:")
remote_con.execute("INSTALL httpfs;")
remote_con.execute("LOAD httpfs;")

def load_remote_table(table_name: str) -> pd.DataFrame:
    query = f"""
    SELECT *
    FROM read_csv_auto('{BASE_URL}/{table_name}.csv.gz')
    """
    return remote_con.execute(query).df()

tables = {name: load_remote_table(name) for name in TABLE_NAMES}

pd.DataFrame(
    {
        "table": list(tables.keys()),
        "rows": [df.shape[0] for df in tables.values()],
        "columns": [df.shape[1] for df in tables.values()],
    }
)

In [ ]:
# 2.2 Helper: snake_case conversion and table unpacking
def to_snake(col: str) -> str:
    s1 = re.sub(r"(.)([A-Z][a-z]+)", r"\1_\2", col)
    s2 = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", s1)
    s3 = s2.replace(" ", "_").replace("/", "_").lower()
    return re.sub(r"__+", "_", s3).strip("_")

for name, df in tables.items():
    df.columns = [to_snake(c) for c in df.columns]

appearances = tables["appearances"].copy()
club_games = tables["club_games"].copy()
clubs = tables["clubs"].copy()
competitions = tables["competitions"].copy()
game_events = tables["game_events"].copy()
games = tables["games"].copy()
player_valuations = tables["player_valuations"].copy()
players = tables["players"].copy()

**Interpretation:** Loading all core tables from one remote source helps avoid mixed local snapshot versions, which is especially important when tables are joined via shared IDs such as `game_id` and `player_id`.

In [ ]:
# 2.3 Parse datetime columns
games["date"] = pd.to_datetime(games["date"], errors="coerce")
player_valuations["date"] = pd.to_datetime(player_valuations["date"], errors="coerce")

if "date" in game_events.columns:
    game_events["date"] = pd.to_datetime(game_events["date"], errors="coerce")

**Note:** Invalid date strings are converted to `NaT`, which makes downstream quality checks explicit instead of silently failing.

In [ ]:
# 2.4 Lightweight validation and target club lookup
required_keys = {
    "players": "player_id",
    "games": "game_id",
    "clubs": "club_id",
}

for table_name, key in required_keys.items():
    missing = tables[table_name][key].isna().sum()
    print(f"{table_name}.{key}, missing values: {missing}")

orphan_player_ids = set(game_events["player_id"].dropna()) - set(players["player_id"].dropna())
print(f"Orphan player_ids in game_events: {len(orphan_player_ids):,}")

club_lookup = clubs.loc[clubs["name"].str.contains("leverkusen", case=False, na=False), ["club_id", "name", "domestic_competition_id"]]
club_lookup

**Interpretation:** Validating key relationships early helps prevent silent join errors, and looking up the target club dynamically is safer than hard-coding IDs into multiple cells.

In [ ]:
# 2.5 Build season-specific subsets
TARGET_CLUB_ID = int(
    clubs.loc[clubs["name"] == TARGET_CLUB_NAME, "club_id"].iloc[0]
)

season_games = games.loc[
    games["date"].between(SEASON_START, SEASON_END)
].copy()

bundesliga_games = season_games.loc[
    season_games["competition_id"] == TARGET_COMPETITION_ID
].copy()

bundesliga_game_ids = set(bundesliga_games["game_id"])

leverkusen_club_games = club_games.loc[
    (club_games["club_id"] == TARGET_CLUB_ID)
    & (club_games["game_id"].isin(bundesliga_game_ids))
].copy()

leverkusen_game_ids = set(leverkusen_club_games["game_id"])

leverkusen_events = game_events.loc[
    (game_events["club_id"] == TARGET_CLUB_ID)
    & (game_events["game_id"].isin(leverkusen_game_ids))
].copy()

leverkusen_events["is_goal"] = (leverkusen_events["type"] == "Goals").astype(int)
leverkusen_events["is_assist"] = leverkusen_events["player_assist_id"].notna().astype(int)

leverkusen_appearances = appearances.loc[
    (appearances["game_id"].isin(leverkusen_game_ids))
    & (appearances["player_club_id"] == TARGET_CLUB_ID)
].copy()

bundesliga_appearances = appearances.loc[
    appearances["game_id"].isin(bundesliga_game_ids)
].copy()

bundesliga_player_ids = set(bundesliga_appearances["player_id"].dropna())
leverkusen_player_ids = set(leverkusen_appearances["player_id"].dropna())

coverage_summary = pd.DataFrame(
    {
        "metric": [
            "Bundesliga games in scope",
            "Leverkusen matches in scope",
            "Leverkusen event rows",
            "Leverkusen appearance rows",
            "Bundesliga players in scope",
            "Leverkusen players in scope",
        ],
        "value": [
            len(bundesliga_games),
            len(leverkusen_club_games),
            len(leverkusen_events),
            len(leverkusen_appearances),
            len(bundesliga_player_ids),
            len(leverkusen_player_ids),
        ],
    }
)

print(f"Target competition: {TARGET_COMPETITION_ID}")
print(f"Target club: {TARGET_CLUB_NAME} ({TARGET_CLUB_ID})")
coverage_summary

**Interpretation:** The rest of the notebook now works from one clearly defined scope, Bayer 04 Leverkusen, Bundesliga, season 2023/24, which makes charts and model results much easier to trust.

## 3. SQL Data Exploration

Re-register the cleaned pandas tables in DuckDB so SQL queries use the exact same season-aligned data used elsewhere in the notebook.

In [ ]:
# 3.1 Register cleaned pandas tables in DuckDB
con = duckdb.connect(database=":memory:")

for name, df in {
    "appearances": appearances,
    "club_games": club_games,
    "clubs": clubs,
    "competitions": competitions,
    "game_events": game_events,
    "games": games,
    "player_valuations": player_valuations,
    "players": players,
}.items():
    con.register(name, df)

con.execute("SHOW TABLES").df()

In [ ]:
# 3.2 Latest market values for Bayer 04 Leverkusen players in Bundesliga 2023/24
latest_val_query = '''
WITH target_games AS (
    SELECT DISTINCT cg.game_id
    FROM club_games cg
    JOIN games g USING (game_id)
    WHERE g.competition_id = ?
      AND g.date BETWEEN ? AND ?
      AND cg.club_id = ?
),
season_players AS (
    SELECT DISTINCT a.player_id
    FROM appearances a
    JOIN target_games tg USING (game_id)
    WHERE a.player_club_id = ?
),
latest_vals AS (
    SELECT
        pv.player_id,
        pv.market_value_in_eur,
        pv.date,
        ROW_NUMBER() OVER (
            PARTITION BY pv.player_id
            ORDER BY pv.date DESC
        ) AS rn
    FROM player_valuations pv
    JOIN season_players sp USING (player_id)
    WHERE pv.date <= ?
)
SELECT
    p.name AS player,
    lv.market_value_in_eur AS latest_market_value_eur,
    lv.date AS valuation_date
FROM latest_vals lv
JOIN players p USING (player_id)
WHERE lv.rn = 1
ORDER BY latest_market_value_eur DESC
LIMIT 10;
'''

latest_val = con.execute(
    latest_val_query,
    [
        TARGET_COMPETITION_ID,
        SEASON_START,
        SEASON_END,
        TARGET_CLUB_ID,
        TARGET_CLUB_ID,
        SEASON_END,
    ],
).df()

latest_val

**Interpretation:** Using the latest valuation on or before season end is usually more robust than averaging sparse valuation snapshots within the season window, and it matches the player scope of the case study.

In [ ]:
# 3.3 Bayer 04 Leverkusen player goals and assists in Bundesliga 2023/24
player_stats_query = '''
SELECT
    p.name AS player,
    SUM(CASE WHEN ge.type = 'Goals' THEN 1 ELSE 0 END) AS total_goals,
    SUM(CASE WHEN ge.player_assist_id IS NOT NULL THEN 1 ELSE 0 END) AS total_assists
FROM game_events ge
JOIN games g USING (game_id)
JOIN players p USING (player_id)
WHERE g.date BETWEEN ? AND ?
  AND g.competition_id = ?
  AND ge.club_id = ?
GROUP BY p.name
ORDER BY total_goals DESC, total_assists DESC
LIMIT 10;
'''

player_stats = con.execute(
    player_stats_query,
    [SEASON_START, SEASON_END, TARGET_COMPETITION_ID, TARGET_CLUB_ID],
).df()

player_stats

**Interpretation:** Restricting the SQL leaderboard to Leverkusen keeps the output aligned with the notebook's case-study narrative and avoids mixing club-specific storytelling with league-wide totals.

## 4. Exploratory Data Analysis (EDA)
Visualize key relationships in the Bayer 04 Leverkusen subset.

### 4.1 Goals vs Assists per Player (2023/24 Season)

In [ ]:
# 4.1 Goals vs assists per player, Bayer 04 Leverkusen, Bundesliga 2023/24
by_player = (
    leverkusen_events
    .groupby("player_id", as_index=False)[["is_goal", "is_assist"]]
    .sum()
    .rename(columns={"is_goal": "goals", "is_assist": "assists"})
    .merge(players[["player_id", "name"]], on="player_id", how="left")
)

minutes_by_player = (
    leverkusen_appearances
    .groupby("player_id", as_index=False)["minutes_played"]
    .sum()
    .rename(columns={"minutes_played": "total_minutes"})
)

by_player = by_player.merge(minutes_by_player, on="player_id", how="left").fillna({"total_minutes": 0})

highlight = (
    pd.concat(
        [
            by_player.nlargest(5, "goals"),
            by_player.nlargest(5, "assists"),
        ],
        ignore_index=True,
    )
    .drop_duplicates("player_id")
)

by_player["is_highlight"] = by_player["player_id"].isin(highlight["player_id"])

plt.figure(figsize=(10, 6))
ax = sns.scatterplot(
    data=by_player,
    x="goals",
    y="assists",
    size="total_minutes",
    sizes=(80, 700),
    hue="is_highlight",
    palette={True: "#C44E52", False: "#4C72B0"},
    alpha=0.75,
    legend=False,
)

texts = []
for _, row in by_player.loc[by_player["is_highlight"]].iterrows():
    texts.append(
        ax.text(
            row["goals"],
            row["assists"],
            row["name"],
            fontsize=9,
            weight="semibold",
        )
    )

adjust_text(texts, arrowprops={"arrowstyle": "-", "color": "gray", "lw": 0.5})

ax.set_title(f"{TARGET_CLUB_NAME}, Goals vs Assists per Player ({SEASON_LABEL})", pad=15, fontsize=14)
ax.set_xlabel("Goals")
ax.set_ylabel("Assists")
plt.tight_layout()
plt.show()

**Interpretation:**  
The scatter shows most Leverkusen players cluster around moderate goal and assist counts, reflecting a collective attacking structure rather than dependence on one outlier scorer. Attackers like Victor Boniface, Florian Wirtz, and Álex Grimaldo stand out for their exceptional combined contributions.

**Actionable Insight:** 
Teams could use this to spot high‐minute, high‐output role players, an approach that Bayer Leverkusen itself has leveraged to maintain consistent performance without over-dependence on a single striker.


## 5. Performance Analysis

### 5.1 Match Result Distribution for Bayer 04 Leverkusen (2023/24)

In [ ]:
# 5.1 Match result distribution for Bayer 04 Leverkusen, Bundesliga 2023/24
leverkusen_club_games["result"] = np.select(
    [
        leverkusen_club_games["own_goals"] > leverkusen_club_games["opponent_goals"],
        leverkusen_club_games["own_goals"] == leverkusen_club_games["opponent_goals"],
    ],
    ["W", "D"],
    default="L",
)

result_counts = (
    leverkusen_club_games["result"]
    .value_counts()
    .reindex(["W", "D", "L"], fill_value=0)
    .rename_axis("result")
    .reset_index(name="count")
)

total_matches = result_counts["count"].sum()
palette = {"W": "#4C72B0", "D": "#55A868", "L": "#C44E52"}

plt.figure(figsize=(8, 5))
ax = sns.barplot(data=result_counts, x="result", y="count", palette=palette)

for patch in ax.patches:
    height = patch.get_height()
    pct = (height / total_matches * 100) if total_matches else 0
    ax.text(
        patch.get_x() + patch.get_width() / 2,
        height + 0.2,
        f"{int(height)}\n({pct:.1f}%)",
        ha="center",
        va="bottom",
        fontsize=10,
    )

ax.set_title(f"{TARGET_CLUB_NAME} Match Results ({SEASON_LABEL})", pad=15, fontsize=14)
ax.set_xlabel("Result")
ax.set_ylabel("Matches")
ax.set_ylim(0, result_counts["count"].max() * 1.15 if total_matches else 1)
plt.tight_layout()
plt.show()

**Interpretation:**  
Bayer 04 Leverkusen's unbeaten domestic campaign, characterized by an overwhelming number of wins and crucial draws, underscores their historic dominance in the 2023/24 Bundesliga season.

**Actionable Insight:** 
Maintaining an unbeaten record across an entire league season suggests not only squad depth but also tactical consistency and strong mental resilience. Other clubs aiming for similar achievements must focus on balancing rotation and maintaining intensity across all competitions.


## 6. Market Value Analysis

### 6.1 Market Value Trend

In [ ]:
# 6.1 Market value trend for players who appeared in the selected Bundesliga season
bundesliga_valuations = player_valuations.loc[
    player_valuations["player_id"].isin(bundesliga_player_ids)
].copy()

bundesliga_valuations["year"] = bundesliga_valuations["date"].dt.year

annual = (
    bundesliga_valuations
    .dropna(subset=["date", "market_value_in_eur"])
    .groupby("year")["market_value_in_eur"]
    .agg(
        p25=lambda s: s.quantile(0.25),
        median="median",
        p75=lambda s: s.quantile(0.75),
        p90=lambda s: s.quantile(0.90),
    )
    .reset_index()
)

annual["year_dt"] = pd.to_datetime(annual["year"].astype(int).astype(str) + "-06-30")

fig, ax = plt.subplots(figsize=(12, 6))

ax.scatter(
    bundesliga_valuations["date"],
    bundesliga_valuations["market_value_in_eur"],
    s=5,
    alpha=0.03,
    color="gray",
)

ax.fill_between(
    annual["year_dt"],
    annual["p25"],
    annual["p75"],
    alpha=0.15,
    label="25th to 75th percentile",
)

ax.plot(annual["year_dt"], annual["median"], marker="o", lw=2, label="Median")
ax.plot(annual["year_dt"], annual["p90"], marker="s", linestyle="--", lw=2, label="90th percentile")

ax.set_yscale("log")
ax.yaxis.set_major_formatter(mtick.StrMethodFormatter("€{x:,.0f}"))
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

ax.set_title("Market Value Trend, Players in Bundesliga 2023/24 Scope", pad=15, fontsize=14)
ax.set_xlabel("Year")
ax.set_ylabel("Market value in EUR")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

**Interpretation:** The trend lines for the median and 90th percentile, overlaid on a background of individual valuations, show how market values evolved for players who appeared in Bundesliga 2023/24.

**Actionable Insight:**  
The distributional spread is often more informative than a single headline value, because it helps separate typical squad-level valuation ranges from the elite end of the market.

## 7. Predictive Modeling

Create a season-level modeling dataset using Leverkusen appearances and the latest available player valuations up to the end of the 2023/24 season.

In [ ]:
# 7.1 Build modeling dataset
season_perf = (
    leverkusen_appearances
    .groupby("player_id", as_index=False)
    .agg(
        total_goals=("goals", "sum"),
        total_assists=("assists", "sum"),
        total_minutes=("minutes_played", "sum"),
    )
)

latest_valuations = (
    player_valuations.loc[player_valuations["date"] <= SEASON_END]
    .sort_values("date")
    .groupby("player_id")
    .tail(1)
    .loc[:, ["player_id", "market_value_in_eur"]]
)

model_df = (
    season_perf
    .merge(latest_valuations, on="player_id", how="inner")
    .dropna(subset=["market_value_in_eur"])
    .assign(log_market_value=lambda df: np.log1p(df["market_value_in_eur"]))
)

model_df.head()

The target is transformed with `log1p`, which usually makes valuation modeling more stable because football market values are heavily right-skewed.

In [ ]:
# 7.2 Define features and target
feature_cols = ["total_goals", "total_assists", "total_minutes"]
X = model_df[feature_cols]
y = model_df["log_market_value"]

### 7.3 Ridge Regression Baseline

In [ ]:
# 7.3 Cross-validated Ridge regression baseline
ridge_model = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=1.0)),
    ]
)

cv_results = cross_validate(
    ridge_model,
    X,
    y,
    cv=5,
    scoring={"r2": "r2", "mae": "neg_mean_absolute_error"},
    return_train_score=False,
)

pd.DataFrame(
    {
        "metric": ["mean_cv_r2", "mean_cv_mae_log_scale"],
        "value": [
            cv_results["test_r2"].mean(),
            -cv_results["test_mae"].mean(),
        ],
    }
)

**Interpretation:** If cross-validated R² remains low or negative, simple counting stats alone are not sufficient to explain player market value. That is still a useful result, because it motivates richer features such as age, position, contract length, xG, expected assists, and injury history.

### 7.4 Random Forest Feature Importance

In [ ]:
# 7.4 Random Forest feature importance
rf_model = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    min_samples_leaf=2,
)

rf_model.fit(X, y)

feature_importance = (
    pd.DataFrame(
        {
            "feature": feature_cols,
            "importance": rf_model.feature_importances_,
        }
    )
    .sort_values("importance", ascending=True)
)

plt.figure(figsize=(9, 5))
bars = plt.barh(feature_importance["feature"], feature_importance["importance"])

for bar in bars:
    plt.text(
        bar.get_width() + 0.005,
        bar.get_y() + bar.get_height() / 2,
        f"{bar.get_width():.2f}",
        va="center",
        fontsize=10,
    )

plt.xlabel("Feature importance")
plt.title("Random Forest, Market Value Drivers from Simple Season Stats", fontsize=13, pad=15)
plt.grid(axis="x", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

**Interpretation:** Feature importance helps describe which simple season-level signals the model relies on most, but it should not be over-interpreted as causal importance.

## 8. Cumulative Attacking Output

### 8.1 Cumulative Goals & Assists Over the 2023/24 Season

In [ ]:
# 8.1 Cumulative goals and assists over the 2023/24 Bundesliga season
leverkusen_daily = (
    leverkusen_events
    .groupby("date", as_index=False)[["is_goal", "is_assist"]]
    .sum()
    .sort_values("date")
)

leverkusen_daily["cum_goals"] = leverkusen_daily["is_goal"].cumsum()
leverkusen_daily["cum_assists"] = leverkusen_daily["is_assist"].cumsum()

plt.figure(figsize=(12, 7))
plt.step(
    leverkusen_daily["date"],
    leverkusen_daily["cum_goals"],
    where="post",
    label="Cumulative goals",
    lw=2,
)
plt.step(
    leverkusen_daily["date"],
    leverkusen_daily["cum_assists"],
    where="post",
    label="Cumulative assists",
    lw=2,
    color="C1",
)

plt.fill_between(
    leverkusen_daily["date"],
    leverkusen_daily["cum_goals"],
    leverkusen_daily["cum_assists"],
    step="post",
    alpha=0.10,
    color="C1",
    label="Assist to goal gap",
)

ax = plt.gca()
ax.xaxis.set_major_locator(mdates.MonthLocator(bymonthday=1, interval=1))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))

plt.title(f"{TARGET_CLUB_NAME}, Cumulative Goals and Assists ({SEASON_LABEL})", pad=15, fontsize=14)
plt.xlabel("Date")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.legend(loc="upper left")
plt.tight_layout()
plt.show()

**Interpretation:**  
The cumulative curves reveal steady and synchronized growth between goals and assists across the season, with noticeable surges during key stretches such as their dominant autumn run and spring title push. This balance between finishing and creativity reflects the tactical intelligence and cohesion Bayer Leverkusen displayed throughout 2023/24.

**Key Moment Spotlight: Title-Race Statement vs. Bayern Munich**

One defining match of Leverkusen's unbeaten season came on **February 10, 2024**, when they beat Bayern Munich **3,0**.  
That result functioned as a title-race statement and fits the broader story shown by the cumulative attacking curves, a team that kept building output without a long performance drop-off.

## Conclusion

**Key takeaways**

1. **Scope clarity matters.**  
   A GitHub-facing notebook is much stronger when every table, chart, and model uses the same clearly defined season and competition scope.

2. **Remote data access improves reproducibility.**  
   Loading one consistent Transfermarkt snapshot through DuckDB is safer than relying on manually assembled local CSV folders.

3. **Leverkusen's season can be told from multiple angles.**  
   Match results, cumulative attacking output, and player contribution charts together create a stronger football narrative than isolated plots.

4. **Simple counting stats are only a starting point for valuation modeling.**  
   Goals, assists, and minutes provide a useful baseline, but realistic market value modeling needs richer player, contract, role, and context features.

---

## References

- Data source: [transfermarkt-datasets by dcaribou](https://github.com/dcaribou/transfermarkt-datasets)
- Remote access pattern: DuckDB + hosted `.csv.gz` files from the project README
- Bundesliga official schedule and match results (2023/24)
- Match results and context: [Bundesliga.com](https://www.bundesliga.com/en), [ESPN FC](https://www.espn.com/soccer/team/_/id/131/bayer-leverkusen)

> The notebook is designed for educational and portfolio purposes. An internet connection is required because the raw data is loaded remotely.